# Delayed-mode QC program for MAPCO2, ASVCO2, and associated sensors

This program was developed by NOAA PMEL for quality control (QC) of delayed-mode data of autonomous air-sea CO2 and ocean acidification observations from buoys and Uncrewed Surface Vehicles (USVs).  The program is designed for QC'ing data from MAPCO2 systems with integrated sensors (SeaBird SBE16 with Aanderaa or SBE63 oxygen optodes and ECO FLNTUS, SAMI pH, and/or SeaFET) and ASVCO2 sensors deployed on Uncrewed Surface Vehicles (USVs). The program will also integrate historical data and independent validation data for use during the QC process.  Once finalized, data and metadata files are generated for upload to SOCAT and data archives.

## Set Up

Before beggining quality control, the user is encouraged to organize the post deployment data and quality control folder structure.
1. **Folder containing raw file(s)** - if using post-deployment MAPCO2 data, there will be one text file. If using ASVCO2 data, there can be one or more netCDF files.
2. **Optional: External data file(s)** - The above folder can also contain any external data files (such as SAMI pH, SeaFET pH, Oxygen, etc.). These files should be formatted as specified in the *\[Optional\] Data Import* section. Any data processing steps (ex salinity compensating oxygen, post-processing pH) must be done before data upload.
3. **Folder for QC data** - This folder will be used by the program to save netCDF files for the various steps of the QC process and a log file and is chosen during the QC procedure. This folder can be placed inside the raw file folder, if the user wishes to keep raw data and QC'd data together. However, the user should not place any netCDF files (ex. ASVCO2 raw files) into the QC data folder. When the QC data folder is first chosen, the program removes all existing netCDF files in the QC data folder structure. This is to allow the user to restart the QC procedure without any residual effects of pre-existing netcdf files.
4. **Optional: Config file** - The user may create a *config.yml* file (having that specific name and extension) before starting the QC procedure, which can be used to specify particular constants and metadata. This file must be placed in the QC data folder. If the user opts to not have this file, the program will automatically generate the file with default constants, which must be changed by the user during the coarse of the QC procedure through GUI interfaces.

## Chlorophyll Climatology
In order to make use of external CHL climatology, the user must use a pre-created a YAML config file that contains a stanza as follows:

<pre>
chl_credentials:
  username: <i>my-username</i>
  password: <i>my-password</i> 
</pre>

where the username and password are provided by request to the user by Globcolour.

## Saving progress
As QC work can take quite some time, a reload feature has been implemented to allow the user to close this jupyter notebook and restart at a later time. The user must have completed the **Map data to SOCAT QC requirements** section in order to use the reload feature. If the user has performed any manual quality control (such as adjusting quality flags and adding quality notes), those changes will be saved as well. Any further changes, such as adding an MBL adjustment, SOCAT metadata, etc. must be redone.

In order to reload, the user can run the **Load previous work** cell and specify the QC data folder when prompted.

## Convert delayed-mode files

If using a MAPCO2 text file, check the file beforehand for the site ID.  PMEL's automatically recognized site IDs are as follows:

<table>
<tr><td>alawai</td><td>boboa</td><td>cb-06</td><td>cce1</td><td>cce2</td></tr>
<tr><td>cheeca</td><td>chuuk_k1</td><td>coastalla</td><td>cola</td><td>coastalmS</td></tr>
<tr><td>crescent</td><td>crimp1</td><td>crimp2</td><td>dabob</td><td>fagatele</td></tr>
<tr><td>gakoa</td><td>graysff</td><td>heron</td><td>hogreef</td><td>iceland</td></tr>
<tr><td>jkeo</td><td>kaneohe</td><td>keo</td><td>kilonalu</td><td>kodiak</td></tr>
<tr><td>enrique</td><td>laparguera</td><td>lapush</td><td>m2</td><td>mosean</td></tr>
<tr><td>ndbcga</td><td>nh</td><td>nh10</td><td>papa</td><td>peggy</td></tr>
<tr><td>sofs</td><td>southeast_ak</td><td>stratus</td><td>tao110w_0n</td><td>110w0</td></tr>
<tr><td>tao125w_0n</td><td>125w0</td><td>tao140w_0n</td><td>140w0</td><td>tao155w_0n</td></tr>
<tr><td>155w0</td><td>tao165e_0n</td><td>165e0</td><td>tao165e_8s</td><td>165e8s</td></tr>
<tr><td>tao170w_0n</td><td>170w0</td><td>twanoh</td><td>wa</td><td>whots</td></tr>
</table></td></tr>

Non-PMEL users can input a different site ID, however the program will not be able to run certain features like ERDDAP historical data download nor will the final SOCAT files have the appropriate site names and details.


### Choose data files
The user must change the site_id in the first cell if it is set to **None**. This id is used to name some files and search for historic data when using the ERDDAP historic data option.

Setting the use_colorblind option to *True* will change the color codes in plots to a colorblind friendlier option. This option does not cover all colorblindness nor monochromacy.

By executing the second cell below, the user will be prompted to select a file by a file selection UI. If QC'ing a MAPCO2 text file or single ASVCO2 netCDF file, navigate to the appropriate file and select. If QC'ing a collection of ASVCO2 netCDF files, navigate to the raw file folder and select one of the files. The other files will automatically be read (*NOTE: make sure all data are present and from the same deployment. No other netCDF files can be present in the folder; the QC procedure is indiscriminate to the netCDF files it attempts to process*).

In [ ]:
import matplotlib.pyplot as plt
%matplotlib ipympl

# optional change
use_colorblind = False

# change this!
site_id = None

In [ ]:
# standard library imports
import pathlib

# local imports
from xco2qc.browse import qt_browse

home = pathlib.Path.home()
mapco2_raw_text_file = qt_browse(home, text='Select the input raw MAPCO2 text file')
print(mapco2_raw_text_file)

### Choose a root directory for original, reduced, and final output netCDF, XML, and CSV files
The user will be prompted to select a QC data folder by a folder selection UI. See Set Up (above) for additional details about this folder. If the user has not already provided a config.yml file in the specified folder, it will be created here.

In [ ]:
# standard library imports
import itertools

# local imports
from xco2qc.file_logging import XCO2SetupLogFile

output_root = qt_browse(home, is_dir=True, text='Select a directory')

print(f'NetCDF data files are to be written to {output_root}')

# We should delete any netCDF files under this root before proceeding further.
for ncfile in output_root.glob('*.log'):
    ncfile.unlink()
for folder in ['raw', 'reduced', 'trimmed', 'merge']:
    if not (output_root / folder).is_dir():
        continue
    for ncfile in (output_root / folder).glob('*.nc'):
        ncfile.unlink()
    
# Log everything to a file under our output root.
log_file = output_root / 'xco2.log'
XCO2SetupLogFile(log_file).run()

### Convert raw data to netCDF

In [ ]:
# This processing step may take some time, especially if a lot of ASVCO2 files are involved.

# local imports
from xco2qc.core import CYCLE_HEADER_NCFILE
from xco2qc.raw_text_conversion import RawTextToRawNC
from xco2qc.span2_coefficient import CheckSpan2Coefficient

raw_netcdf_dir = output_root / 'raw'
with RawTextToRawNC(
    mapco2_raw_text_file, raw_netcdf_dir, verbosity='INFO', deployment_number=None
) as p:
    p.run()

ncfile = raw_netcdf_dir / CYCLE_HEADER_NCFILE
with CheckSpan2Coefficient(ncfile) as p:
    p.run()

## [Optional, Multi-run] External Data Import¶

The user may optionally import MET (SST and/or SSS), SAMI data, SBE63 data, and/or SeaFET data here, regardless of whether it was processed above. If there is more than one data type to import, i.e. MET and SAMI, run the code block more than once, selecting a new external data source file.

### Notes
1. All import files are expected to be text, tab-delimited files.
2. If choosing to import, be aware that the imported data will take precedence over the data in the text file uploaded previously or in the MAPCO2/ASVCO2 raw files.
2. The following formats, including column names, are necessary for the QC procedure to recognize data correctly. If the imported files are not correct, the data may not be ingested correctly.

### MET

The data may be imported from a CSV or TXT file that is either comma-delimited or tab-delimited (not space-delimited), i.e.

<pre>
time,salinity,temperature                                       
2020-05-23 12:00:00,33,22                                       
2020-05-23 12:30:00,35,23 
.
.
.
</pre>

In addition, the user may import salinity and temperature separately.  Valid names for the salinity and temperature variables include **SSS**, **salinity**, **SST**, **temp**, and **temperature**.

### SAMI

The external SAMI file format is tab-delimited and is required to follow this example. The columns are the standard output columns from the SAMI client post processing program, however the header may be different depending on the client version. This program currently only accepts the header specified below, so the user may need to adjust their SAMI file accordingly.

<pre>
SAMI2-pH
Seawater
DateTime         SAMIpH  pHerror             TempC       Salinity        Flags(Outlier,Pump,Sat,Blank)
09/24/2018 03:03:31            8.0634         0.00139         26.6749         34.8600    0000 
09/24/2018 06:03:31            8.0663         0.00206         26.6212         34.8683    0000 
09/24/2018 09:03:31            8.0641         0.00263         26.5407         34.8613    0000
.
.
.
</pre>

### SBE63

The external SBE63 file CSV file format is tab-delimited and required to follow this example:

<pre>
hdrtime         sc_o2_umolkg 
2014-07-17 05:30:00     206.082115407291 
2014-07-17 06:00:00     205.869220684994 
2014-07-17 06:30:00     205.644511829777 
.
.
.
</pre>

### SeaFET

the external SeaFET CSV file format is comma-delimited and required to follow this example:

<pre>
SATFHR,FILE-VERSION,3.0.0
SATFHR,APP-VERSION,SeaFETCom 2.0.2_106
SATFHR,OPERATOR,musielewicz
SATFHR,TIME-STAMP,2018-07-20T10:43:39-0700
SATFHR,SERIAL-NUMBER,0094
SATFHR,CAL_PHINT_OFFSET_COEFF,-8.330500e-02
SATFHR,CAL_PHINT_SLOPE_COEFF,-1.101000e-03
SATFHR,CAL_PHEXT_OFFSET_COEFF,-1.103161e+00
SATFHR,CAL_PHEXT_SLOPE_COEFF,-1.048000e-03
SATFHR,ONBOARD_SALINITY_PSU,35.000
SATFHR,SAMPLES_AVERAGED,1
SATFHR,RAW-DATA-FILE-NAMES,C0000001.CSV

Frame Header,DATE TIME,PH_INT,PH_EXT,TEMP,NEW_PH_INT,NEW_PH_EXT,PRO_TEMP,PRO_SALINITY,PRO_PRESSURE,CTD_OXYGEN,STATUS,CHECK
SATPHP0094,12-Aug-2015 15:41:25.080,6.7202,6.6050,22.43,6.6933,6.5474,29.38,33.937,,,0,242
SATPHP0094,12-Aug-2015 15:41:29.039,6.7207,6.6054,22.43,6.6938,6.5478,29.38,33.937,,,0,216
SATPHP0094,12-Aug-2015 15:41:33.000,6.7212,6.6061,22.43,6.6943,6.5485,29.38,33.937,,,0,245
.
.
.
</pre>

In [ ]:
# local imports
from xco2qc.external_data import ImportExternalData

datafile = qt_browse(home, text='Select the external MET/SAMI/SBE63 text file')
print(datafile)

with ImportExternalData(datafile, raw_netcdf_dir, verbosity='INFO') as p:
    p.run()

## [Optional] Import Historical Data via ERDDAP

Run this step to optionally import historical data that has already been QC'd from the PMEL ERDDAP server.

*If there is a KeyError, check the site_id in the first live code cell (Choose data files section), replace with a recognized name in the table under "Convert delayed-mode files", then rerun only that cell. This cell can then be rerun.*

In [ ]:
# local imports
from xco2qc.external_historical import ImportHistorical

with ImportHistorical(None, raw_netcdf_dir, site_id=site_id, verbosity='INFO') as p:
    p.run()

## [Optional] Import Historical Data in CSV Format

As an alternative to importing via ERDDAP, run this optional step to import historical data in CSV or TXT format (**must be tab-delimited**).  The historical data should already be QC'd. The file must have a datetime column followed by one or more data variables.  Permissable data column names (case sensitive) are: chl, doxy, fCO2_air, fCO2_sw, ntu, pCO2_air, pCO2_sw, pH_sw, SSS, SST, xCO2_air, xCO2_sw. 
 See the following example.

<pre>
datetime	SST
2006-10-16 21:00	19.872
</pre>

The user may run this cell block multiple times with different data streams.

In [ ]:
# local imports
from xco2qc.external_historical import ImportHistorical

historical_file = qt_browse(home, text='Select the historical text file')
print(historical_file)

with ImportHistorical(historical_file, raw_netcdf_dir, verbosity='INFO') as p:
    p.run()

# [Optional]  Import validation data

Import validation data. Unlike historical data, which occurs before the deployment being QC'ed, validation data is data that cooccurs with the deployment. Since validation data is not as often accessible, these parameters will not be automatically plotted. The user can add plots during the manual QC process step: *[Optional] Manual QC Step #3: Quality Control Plots*

The file must have a datetime column followed by one or more data variables.  Permissable data column names (case sensitive) are: chl, chl_nighttime, dissolved_oxygen, fCO2_air, fCO2_sw, ntu, pCO2_air, pCO2_sw, pH_sw, SSS, SST, xCO2_air, xCO2_sw.

In [ ]:
# local imports
from xco2qc.validation_data import ValidationData

validation_file = qt_browse(home, text='Select validation file')
print(validation_file)

with ValidationData(validation_file, raw_netcdf_dir, verbosity='INFO') as p:
    p.run()

## SBE16 Auxiliary Sensors

Running the next cell will create a GUI for the user to identify auxilliary sensor channels and calibration information. This cell must be run, but the user can uncheck the *Process Auxiliary sensors* checkbox if no auxiliary sensors exist or do not need QC'ing.

In [ ]:
from xco2qc.aux_sensor_gui import AuxSensorGui

reduced_dir = output_root / 'reduced'

sbe16gui = AuxSensorGui(src_dir=raw_netcdf_dir, dst_dir=reduced_dir)
sbe16gui.run()

## Data Reduction

This next step takes the raw netCDF measurements/arrays and reduces them to 1D variables by averaging.  If the user would prefer using the median, the mapco2_licor_reduce variable can be changed from REDUCE_MEAN to REDUCE_MEDIAN.

If the user requires temperature and/or salinity to post-process other data (ex. pH) using an external program, a CSV file of temperature and salinity is produced during this Data Reduction step. The CSV file  is stored in the reduced folder of QC data folder previously specified by the user.  Post-processed, external data may be imported in the previous "\[Optional] External Data Import" step. Since new data has been added, the "Data Reduction" steps would need to be rerun as well.

In [ ]:
# local imports
from xco2qc.core import REDUCE_MEAN, REDUCE_MEDIAN
from xco2qc.data_reduction import XCO2Reduce
from xco2qc.span2_coefficient import FixSpan2Coefficient

kwargs = sbe16gui.gather_kwargs()

with XCO2Reduce(
    raw_netcdf_dir, reduced_dir,
    verbosity='INFO', mapco2_licor_reduce=REDUCE_MEAN,
    **kwargs
) as p:
    p.run()

ncfile = reduced_dir / CYCLE_HEADER_NCFILE
span2_replacement_value = None
with FixSpan2Coefficient(ncfile, span2=span2_replacement_value, span_correction=False) as p:
    p.run()

## Calculate xCO2 (dry), pCO2, and O2 concentration


In [ ]:
# local imports
from xco2qc.pre_xco2_processing import PreXCO2Calc
from xco2qc.o2_concentration import CalcO2Concentration

with PreXCO2Calc(reduced_dir, verbosity='INFO') as p:
    p.run()
    
with CalcO2Concentration(reduced_dir, verbosity='INFO') as p:                       
    p.run()  

# Initial Plots

The following cell generates a series of plots including data from the deployment being QC'd and some historical data.  By visualizing the full time series, the user can identify anomalies and get a general sense of data quality.  Control the width and height of each plot with the figsize parameter if necessary. Each figure is independent and can be zoomed and panned as necessary. If the user has not interacted with this style of plots before, a brief review of the figure controls can be found here: https://matplotlib.org/stable/users/explain/figure/interactive.html#default-ui. The subplot configuration will not be available. Controls for individual figures will appear when the user hovers over a figure.

Uncomment (remove preceding #s) from the middle lines to change the jupyter notebook size to use the full width of the window.

In [ ]:
from xco2qc.static_initial_summary_plots import StaticInitialSummaryPlots

import pathlib

import seaborn as sns

# %matplotlib ipympl

# from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))

sns.set()
figsize=(10,6)
with StaticInitialSummaryPlots(reduced_dir, figsize=figsize, use_colorblind=use_colorblind, verbosity='INFO') as p:
    p.run()

# [Optional] Other Initial Plots

Execute the cell below to produce a GUI that allows the user to create additional plots of various parameters.

These plots have a zoom in and out feature.

To create new plots, run the entire cell again instead of just selecting new variables.  Control the width and height of the plots with the figsize parameter if necessary.

In [ ]:
# # local imports
# from xco2qc.initial_summary_gui import InitialSummaryGUI

# with InitialSummaryGUI(reduced_dir, figsize=(12, 6)) as p:
#     p.run()

## Trim Deployment Dates

Run the next cell to create a GUI for setting the start and end dates of the deployment if needed (e.g., if pre/post-deployment data is included in the data set).  Use the calendar widgets to select a date and time.

The user is encouraged to run the trimming without touching the GUI at least once.  The *Trim Data and Display Results* section will produce a plot of the MAPCO2 mode variable, which may be helpful in determining a good starting and ending point to the deployment. The | vertical lines indicate the start and end trim dates. Data indicated in red will be trimmed.

### Display Trim GUI

In [ ]:
# local imports
from xco2qc.core import TRIMMED_NCFILE
from xco2qc.trim_time_gui import TimeGUI

# %matplotlib ipympl

with TimeGUI(reduced_dir) as timegui:
    timegui.run()

### Trim Data and Display Results

In [ ]:
# local imports
from xco2qc.trim_netcdf import TrimXCO2netCDF
from xco2qc.post_trim_plots import XCO2PostTrimPlots

# %matplotlib ipympl
trimmed_dir = output_root / 'trimmed'

trim_kwargs = timegui.gather_kwargs()
print(trim_kwargs)
with TrimXCO2netCDF(
    reduced_dir, trimmed_dir, verbosity='INFO', **trim_kwargs
) as t:
    t.run()
    
with XCO2PostTrimPlots(reduced_dir, trimmed_dir, use_colorblind=use_colorblind) as p:
    p.run()

## Manual QC Step #1:  Flag drifting or off-station data

Execute the cell below to display a map of GPS locations.

The red symbol is the center of all current latitude/longitude data (this may be more useful in the case of a stationary mooring than for an autonomous vehicle). Use the mouse to draw a polygon the around points to mask/flag outlying data.  A polygon may be closed by clicking on the first polygon vertex.  When data are masked, the center is recomputed from the updated lat/lon pairs.

As failures in GPS fixes do not necessarily reflect on the quality of the associated data, GPS points that are manually flagged here will not affect the SOCAT quality flag of the associated data.

Use the zoom-to-rectangle button to zoom in to a particular region but make sure that the button is unselected before trying to mask another set of points.

Masked data will not be shown on the figure. The user may **unmask** points by drawing another polygon around previously masked points. If you wish to quit the current polygon selection, hit the ESC key.

The finalized form of this data will depend on the type of platform (which the user will specify later in this process). Stationary platforms will use the averaged latitude and longitude of the entire deployment. Moving platforms will use the recorded latitudes and longitudes and linearly interpolate flagged GPS fixes.

In [ ]:
# 3rd party library imports

# local imports
from xco2qc.manual_qc_gps import ManualGpsQC
from xco2qc.core import CYCLE_HEADER_NCFILE

# %matplotlib ipympl

ncfile = trimmed_dir / CYCLE_HEADER_NCFILE
with ManualGpsQC(ncfile, verbosity='INFO') as p:
    p.run()

## Manual QC Step #2:  LICOR Temperature - Span Coefficient Regression

In this section, span gas coefficients for the MAP/ASVCO2 are derived from the linear regression between the calibration coefficients and the corresponding LI-820/830 temperature measurements acquired during the span cycle over the course of the deployment. This post-deployment reprocessing facilitates the accurate calculation of xCO2 (wet) values from the raw detector counts when rare miscalibrations occur, resulting in erroneous coefficients during the deployment.  This regression can also be applied to generate span gas coefficients if span gas ran out during the deployment.

Run the cell immediately below to display a slider controlling the number of temperature / span coefficient clusters.  Before trying to adjust the slider, run the cell in *Manual QC: display regression plot* to see how many clusters there are. If more than one cluster are necessary, adjust the slider and rerun the *Manual QC: display regression plot* cell.

After the plot window has opened, use the mouse to select a polygon around anomalous points to exclude from the regression calculations.  Use the zoom-to-rectangle button to zoom in to a particular region but make sure that the zoom button is deselected before trying to mask another set of points. Exit out of the current polygon selection (without starting over) by hitting the ESC key.

Temperatures where span was lost (span flag raised) are indicated by black markers at the bottom of the plot. These points do not need to be selected as anomalous.

### Manual QC:  select number of clusters

In [ ]:
plt.close('all')

# 3rd party library imports
import ipywidgets

n_clusters_slider = ipywidgets.IntSlider(value=1, min=1, max=4)
display(n_clusters_slider)

### Manual QC:  display regression plot

In [ ]:
# local imports
from xco2qc.manual_qc_regression import ManualRegressionQC
from xco2qc.core import MODELS_FILE

models_file = reduced_dir / MODELS_FILE
with ManualRegressionQC(
    trimmed_dir, n_clusters=n_clusters_slider.value, use_colorblind=use_colorblind, verbosity='INFO',
    models_file=models_file
) as p:
    p.run()

## Recalculate xCO2 (wet, dry)

Run the next two cells to recalculate xCO2 based on the modified LICOR Temperature - Span Coefficient Regression. This typically corrects the occasional xCO2 measurement that has a bad calibration. The version of the Licor is dependent on the type of licor present and the firmware version for Licor 820s. ASVCO2s are currently always Licor 830s. MAPCO2s will have Licor 820s with one of two firmwares. As a general rule of thumb, if there is a span 2 coefficient present in the raw data, the licor 820 will be a version 2.

The user can choose not to calculate post-xCO2 if the results of the LICOR regression above are unsatisfactory. 

### Select whether to recalculate xCO2

In [ ]:
from xco2qc.post_xco2_gui import PostXCO2GUI

xco2_gui = PostXCO2GUI()
xco2_gui.run()

plt.close('all')

### Recalculate xCO2

In [ ]:
# local imports
from xco2qc.post_xco2_processing import PostXCO2Calc
from xco2qc.post_xco2_plots import PostXCO2Plots
import matplotlib.pyplot as plt

plt.close('all')

recalc_kwargs = xco2_gui.gather_kwargs()

with PostXCO2Calc(
    trimmed_dir, verbosity='INFO', **recalc_kwargs
) as p:
    p.run()

with PostXCO2Plots(trimmed_dir, use_colorblind=use_colorblind, verbosity='INFO') as p:
    p.run()

# Automatic QC: Flag xCO2 data algorithmically
Run the next cell to display the automatic QC setup GUI.  The GUI allows the user to select the range of good values for key MAPCO2 and ASVCO2 system diagnostics.  Values outside of these ranges will be automatically flagged in the following step.  The user can run the cell to perform the actual QC, then go back and adjust the settings in the GUI if needed.

This can take much longer to complete if spike detection is turned on. 

After the automatic QC is finished, the program will generate a log of the QC calculations, plots of the results of each QC check, and a table of all the QC flag detections. The program will also update the config.yml file to reflect changes in the GUI. This is to allow the user to rerun the QC procedure with the same values/ranges.

### Display the Automatic QC Setup GUI

Above the GUI, a plot of the pump off - on differences is displayed to help the user choose appropriate ranges.

**Note:** For the span pressure difference range, the plot should show negative differences, but the ranges are selected as positive values. For example, if the span pump difference plot shows a data range of -1.5 to -2.5 with some outliers at -3.5 and below, the user can set the span pump difference range to 1.5 and 2.5 to flag the outliers.

In [ ]:
from xco2qc.qc_gui import QCGui

figsize=(10,6)
with StaticInitialSummaryPlots(reduced_dir, figsize=figsize, use_colorblind=use_colorblind, verbosity='INFO') as p:
    p.plot_pump_qc()

gui = QCGui(dst_dir=trimmed_dir)
gui.run()

### Perform Automatic QC

In [ ]:
plt.close('all')

# local imports
from xco2qc.update_config import update_config_file
from xco2qc.qc import QCChecker
from xco2qc.qc_plots import QCPlots
from xco2qc.qc_statistics import XCO2QCStatistics

# update config file
aux_kwargs = sbe16gui.gather_kwargs()
qc_kwargs = gui.gather_kwargs()
update_config_file(output_root, aux_kwargs, qc_kwargs)

with QCChecker(trimmed_dir, verbosity='INFO', **qc_kwargs) as p:
    p.run()
    
with QCPlots(trimmed_dir, verbosity='INFO', **qc_kwargs) as p:
    p.run()

with XCO2QCStatistics(ncfiles=trimmed_dir) as p:
    p.run()

## Auxiliary pH Calculation

Identify the geographic region for the deployment, which will determine equations to estimate Total Alkalinity (TAlk). This cell must be run, select *None* for no calculation.

In [ ]:
from xco2qc.pco2sys_gui import PCO2SYS_Region_Gui

with PCO2SYS_Region_Gui() as pco2sys_obj:
    pco2sys_obj.run()

## Perform Data Merge

Merge deployment data processed to this point into a single netCDF file.

In [ ]:
# local imports
from xco2qc.core import MERGE_NCFILE
from xco2qc.merge import XCO2Merge

region=pco2sys_obj.gather_kwargs()['region']

merge_dir = output_root / 'merge'
merge_ncfile = merge_dir / MERGE_NCFILE
with XCO2Merge(trimmed_dir, merge_ncfile, verbosity='INFO', region=region) as p:
    p.run()

## Map data to SOCAT QC requirements

Translate QC bitmasks into discrete SOCAT QC values.

In [ ]:
# local imports
from xco2qc.socat_qc import SocatQC

with SocatQC(ncfile=merge_ncfile, verbosity='INFO') as p:
    p.run()
    config = p.config

## [Optional] Load previous work
If the user has restarted the notebook and would like to continue where they left off, run the following code block. User must have completed the "Map data to SOCAT QC requirements" to load their previous work. If the user has not completed that step, they must restart the process. Make sure to adjust site_id and optionally use_colorblind so that they match the start of the QC procedure.

The user will be prompted with a folder dialog UI. Select the appropriate QC data folder originally specified in the **Choose a root directory for original, reduced, and final output netCDF, XML, and CSV files** section to reload the data.

In [ ]:
import pathlib
import matplotlib.pyplot as plt
%matplotlib ipympl

# local imports
from xco2qc.file_logging import XCO2SetupLogFile
from xco2qc.browse import qt_browse
from xco2qc.load_project import load_project

home = pathlib.Path.home()
output_root = qt_browse(home, text='Select a directory', is_dir=True)
trimmed_dir, merge_dir, models_file, merge_ncfile, config = load_project(output_root)
qc_kwargs = config["QC"]
print(output_root)

# Log everything to a file under our output root.
log_file = output_root / 'xco2.log'
XCO2SetupLogFile(log_file).run()

# change this!
site_id = None
use_colorblind = False

## [Optional] Manual QC Step #3: Quality Control Plots

Run the cell below to display standard QC plots GUI. By clicking "Plot Quality Control", plots will be displayed in separate windows. Additional plots can be added via the GUI and will always be added to the data html page. The html pages will need to be regenerated to show new plots by clicking the "Plot Quality Control" button. These plots persist as html files in the merge folder of the QC root folder and can be reopened at a later time. The "Plot Multi-Year" button will organize major QC variables by year and plot those data by day of the year.

For more complex plotting capabilities, the user can click "Data to Excel", which will generate an Excel workbook with data from the various trimmed, merged, and historic data sources.

Though this cell is not strictly necessary to the QC process, the user is highly recommended to generate the QC plots and use them as a guide to help quality control data. Time series plots are linked and will align themselves when one is zoomed or panned. The data, diagnostic, map and Multi-year html pages cannot be linked, so they will have to be individually panned and zoomed. Automatically flagged data are marked with a black and yellow x and by hovering over said points, the reason for the flag will be shown. Plot controls are at the top right of the page. Data can be turned off/on by clicking on the parameter name list on the right side of the page.

Additional manual flags can be applied to the data via the GUI described in the next section (Apply additional flags).

In [ ]:
# %matplotlib ipympl
plt.close('all')

# local imports
from xco2qc.add_plot_widget import AdditionalPlotsGUI

with AdditionalPlotsGUI(
        merged_folder_path=merge_dir,
        trimmed_folder_path=trimmed_dir,
        site_id = site_id,
        config = config,
        use_colorblind=use_colorblind,
        isMobile=True
) as gui:
    gui.run()

## Manual QC Step #3: Apply additional flags

Run the cell below to apply manual flags. The primary parameter for manual QC flagging will be displayed in the first plot and can be changed using the avaialable drop down. Values in that plot can be flagged good/questionable/bad or good/bad (depending on the parameter) by selecting the type of flag and drawing a polygon around the value(s). The user should provide a quality control note providing an explanation of the QC decision. Questionable and bad data, with associated notes, will be added to the SOCAT QFLog output file. Exit out of a polygon selection by hitting the ESC key.



In [ ]:
from xco2qc.final_qc_gui import XCO2FinalQCGUI

qflog = merge_dir / 'qflog.csv'
with XCO2FinalQCGUI(merge_ncfile, trimmed_dir, qflog=qflog, use_colorblind=use_colorblind, verbosity='INFO') as gui:
    gui.run()

## Determine MBL Adjustment

By comparing the offset between the historical data and the global atmospheric xCO2 marine boundary layer (MBL) data product, we can determine whether a constant offsets due to unknown internal issues in MAPCO2 instruments exists in the deployment being QC'ed. This is only possible if there is enough historical data to determine a trend (ex. the particular site may be perennially low compared to the MBL product, so we might expect the current deployment to also be low).

If retrieve_remote is set to true, the notebook will attempt to retrieve the latest MBL data from NOAA ESRL based on the coordinates in the MAP/ASVCO2 data file, which could take a minute or so.  Otherwise, a cached and possibly out of date version of the MBL data will be used.

In [ ]:
# local imports
from xco2qc.mbl import CompareMBL
from xco2qc import core

# %matplotlib ipympl

retrieve_remote = False
historical_ncfile = output_root / 'reduced' / core.HISTORICAL_NCFILE

with CompareMBL(
    src_dir=trimmed_dir,
    merge_ncfile=merge_ncfile,
    historical_ncfile=historical_ncfile,
    retrieve_remote=retrieve_remote,
    use_colorblind=use_colorblind
) as p:
    p.run()

## Apply MBL, Licor Adjustments

Any adjustment determined from reviewing the MBL time series can be applied here.  If there is a pre-determined adjustment to Licor pressure based on lab quality assurance procedures, that can also be applied here.  If no adjustments are to be made, run this code cell with both parameter values **licor_pressure_adjustment** and **mbl_correction** in the GUI set to zero.

In [ ]:
from xco2qc.mbl_licor_gui import MBLLicorGui

mbl_licor_gui = MBLLicorGui(dst_dir=trimmed_dir)
mbl_licor_gui.run()

In [ ]:
# local imports
from xco2qc.adjustments import XCO2Adjustments

mbl_licor_kwargs = mbl_licor_gui.gather_kwargs()

with XCO2Adjustments(
    trimmed_dir, merge_ncfile,
    mbl_correction=mbl_licor_kwargs['mbl_correction'],
    pressure_correction=mbl_licor_kwargs['licor_pressure_correction'],
    verbosity='INFO'
) as p:
    p.run()

## [Optional] Compute QC Statistics

Display the detection count of each QC category.  It is possible that a single data point can have more than one 
QC flag, so the sum of the category counts might not sum to the length of the time series.

In [ ]:
from xco2qc.qc_statistics import XCO2QCStatistics

with XCO2QCStatistics(merge_ncfile) as p:
    p.run()

# Enter Metadata

If you need to document more than five PIs, change the value of *n* below accordingly and rerun the cell - any changes made in the GUI before rerunning will be lost.

In [ ]:
from xco2qc.metadata_gui import MetadataGUI

n = 5
mgui = MetadataGUI(num_pis=n, netcdf_dir=trimmed_dir)
mgui.run()

## Apply Metadata Conventions

Apply CF, ACDD metadata conventions to the final netCDF file.

In [ ]:
# local imports
from xco2qc.metadata_conventions import MetadataWriter

metadata_kwargs = mgui.gather_kwargs()

with MetadataWriter(merge_ncfile, verbosity='INFO', **metadata_kwargs) as m:
    m.run()

## Generate SOCAT Output
Running the following code will produce the SOCAT output files and save them in the *merge* folder of the QC data folder selected previously. The three files are a data csv, QFLog csv, and a metadata xml. Each file will be named similarly, starting with the station name, start date, end date. If the site_id specified by the user is not one of the cannonical sites, the station name will default to 'unknown' and the Mooring name column in the data csv will default to 'unknown'.

The files will be located in the merge folder of the user specified root directory.

In [ ]:
# local imports
from xco2qc.socat import SocatWriter

metadata_kwargs = mgui.gather_kwargs()

output_xml_folder = output_root / 'merge'
output_csv_folder = output_root / 'merge'

with SocatWriter(
    ncfile=merge_ncfile,
    xmlfile=output_xml_folder,
    csvfile=output_csv_folder,
    licor_pressure_correction=mbl_licor_kwargs['licor_pressure_correction'],
    initial_span_cal=qc_kwargs['initial_span_cal'],
    verbosity='INFO',
    models_file=models_file,
    qflog=qflog,
    **metadata_kwargs
) as p:
    p.run()

## [Optional] Generate Report
Running the following code will generate a word document summarizing the QC process. This report, named '{Station Name} {Deployment Number} Summary.doc', will be located in the merge folder of the selected root directory. This report is standardized and not exhaustive. Any additional information (such as additional plots and explanations) will need to be added to the word document by the user.

In [ ]:
# %matplotlib ipympl
import matplotlib.pyplot as plt
from xco2qc.report_writer import ReportWriterWidget

plt.close()
with ReportWriterWidget(output_root, site_id, use_colorblind=use_colorblind) as writer:
    display(writer.run())

## [Optional] Generate QC Summary

In [ ]:
from xco2qc.summary import XCO2Summary

%matplotlib inline

with XCO2Summary(merge_ncfile, use_colorblind=use_colorblind) as p:
    p.run()